# 📦 Data Shipping 101: A Beginner's Guide to Protocol Buffers (Protobuf)

Welcome! If you've never heard of "Data Encoding" or "RPC," don't worry. Think of this notebook as a guide to running an **International Courier Service**.

### The Big Idea: Efficient Cross-Border Communication
Imagine you have a **Chinese Chef (a Python program)** and a **German Customer (a Java program)**. They need to exchange recipes, but they don't speak the same language, and sending huge, heavy books (like slow JSON files) is too expensive.

**Protocol Buffers (Protobuf)**, developed by Google, is another powerful tool that solves this by:
1.  **Providing a Contract**: A standard "package specification" everyone understands.
2.  **Vacuum Packing**: Compressing the data into a tiny binary format that's super cheap and fast to ship.
3.  **Universal Translator**: Generating code in many languages from a single specification.

## 1. What is Protocol Buffers (Protobuf)?

Protobuf is a language-neutral, platform-neutral, extensible mechanism for serializing structured data. It's like a highly efficient, automated way to pack and unpack information.

It supports basic types like:
- `double`, `float`: Numbers with decimal points
- `int32`, `int64`: Whole numbers (integers)
- `string`: Text
- `bool`: True/False
- `bytes`: Raw data

We define these in a `.proto` file, which acts as our **Package Specification** or **Contract**.

## 2. Step 1: The Package Specification (The Schema)

Before we ship anything, we need to agree on what a "Person" package looks like. 

**The Analogy: The Menu Numbers**  
Just like in a restaurant, instead of saying "I want a double-patty beef burger with extra cheese and no onions," you just say **"I'll take a Number 1."** It's much faster. 

In the code below, notice the numbers `= 1;`, `= 2;`, and `= 3;`. These are our **Field Tags** (Menu Numbers). Protobuf uses these numbers to identify data instead of using long names, making it incredibly fast and efficient for data transfer.

Also notice `repeated string interests = 3;`. `repeated` means this field can appear multiple times, essentially creating a list. It's like saying "you can have many interests, and each one will be tagged as #3."

In [1]:
%%writefile ../schema/person.proto
syntax = "proto3";

message Person {
  string user_name = 1;
  optional int64 favorite_number = 2;
  repeated string interests = 3;
}

message CourseRequest {
  Person person = 1;
  string course = 2;
}


service School {
  rpc teachCourse(CourseRequest) returns (Person) {}
}

Writing ../schema/person.proto


## 3. Step 2: The Universal Translator (Code Generation)

Once we have our `.proto` file (our package specification), we need to turn it into actual code that our Python programs (or Java, Go, etc.) can understand and use.

**The Analogy: The Universal Translator Machine**  
The `protoc` command is like a special machine that reads our `.proto` specification and automatically generates all the necessary code (classes, methods) in the language we choose (Python in this case). This means we don't have to manually write code to handle the packing and unpacking of our `Person` data or the `School` service calls.

We'll run this command in your terminal. It will create two Python files: `person_pb2.py` (for the data structures) and `person_pb2_grpc.py` (for the service communication).

Then, run the following command in a new terminal to generate the Python code:

- Use the cell below to copy the command and paste on a new terminal.
- Make sure your terminal are in the folder `5m-data-2.3-data-encoding-creation-flow`
- Make sure you are in the right environment using `conda activate bde`.

This will generate the following files:

```bash
person_pb2.py
person_pb2_grpc.py
```

## 4. Step 3: The Kitchen (The Server)

Now we need someone to actually *do* something with the data. 

**The Analogy: The Chef in the Processing Center**  
The Server is like a Chef in a kitchen or a processing center. They sit there waiting for an order (an RPC call). When an order comes in, they follow the instructions (like adding a course) and send the result back.

In our code, the `teachCourse` method is the Chef's skill: they take a `Person` (from the `CourseRequest`), add a new `course` to their `interests` list, and return the updated `Person`.

In [2]:
%%writefile ../person_protobuf_server.py
from concurrent import futures
import grpc
import person_pb2_grpc


class School(person_pb2_grpc.SchoolServicer):

  def teachCourse(self, request, context):
    # The 'Chef' adds a new skill to the person's interests
    request.person.interests.append(request.course)
    return request.person

server = grpc.server(futures.ThreadPoolExecutor(max_workers=2))
person_pb2_grpc.add_SchoolServicer_to_server(
    School(), server)
server.add_insecure_port('[::]:50051')
print("Chef is in the kitchen (Server started)...")
print("Press Ctrl+C to stop the server.")
server.start()
server.wait_for_termination()

Writing ../person_protobuf_server.py


Then, run `python person_protobuf_server.py` in a new terminal. This will start the server.  

- Use the cell below to copy the command and paste on a new terminal.
- Make sure your terminal are in the folder `5m-data-2.3-data-encoding-creation-flow`
- Make sure you are in the right environment using `conda activate bde`.

> If the run is successful, you will see "Chef is in the kitchen (Server started)..." and then "Press Ctrl+C to stop the server.". There will be no further prompt until you stop it.

## 5. Step 4: The Customer (The Client)

Now we act as the customer. We will create a person and send them to the kitchen to learn something new.

**The Analogy: Placing the Order**  
When you call `stub.teachCourse(...)`, it feels like you're doing it locally on your computer. But behind the scenes, Protobuf is **packing** your data into a tiny binary package, **shipping** it to the server (the Chef), and **bringing back** the result. This is what Remote Procedure Call (RPC) means – calling a function on a different computer as if it were local.

In [5]:
import sys
sys.path.append('..') # This helps Python find the generated protobuf files
import grpc
import person_pb2
import person_pb2_grpc


In [6]:
def teach_course(stub, person, course):
    # Create a CourseRequest package with the person and the course
    request = person_pb2.CourseRequest(person=person, course=course)
    # Send the request to the server and get the updated person back
    updated_person = stub.teachCourse(request)
    return updated_person


# Connect to the kitchen (server) on localhost:50051
with grpc.insecure_channel('localhost:50051') as channel:
    # Create a person named Martin
    martin = person_pb2.Person(
        user_name='Martin', favorite_number=1337, interests=["daydreaming", "hacking"]
    )
    print(f"Martin's initial interests: {martin.interests}")

    # Create a 'stub' which is like our direct line to the School service
    stub = person_pb2_grpc.SchoolStub(channel)

    # Send Martin to school to learn 'coding' remotely!
    martin = teach_course(stub, martin, "coding")

    # Let's see if he learned it
    print(f"Martin's updated interests: {martin.interests}")

Martin's initial interests: ['daydreaming', 'hacking']
Martin's updated interests: ['daydreaming', 'hacking', 'coding']


## 🎓 Beginner Exercises

Try these to see if you've mastered the "Courier Service":

1.  **Update the Package Specification**: Go back to the `.proto` cell. 
    *   Add a new field `grade` (an integer between 0-100) with an appropriate type annotation (e.g., `int32`) to the `Person` message. Give it a new unique field tag (e.g., `= 4`).
    *   Define a new `message GradeRequest` that contains a `Person` and an `int32 grade`.
    *   Add a new method `assignGrade` to the `School` service that takes a `GradeRequest` and returns the updated `Person`.
2.  **Re-translate the Specification**: Run the `protoc` command again in your terminal to generate updated Python files.
3.  **Update the Chef**: Modify the `person_protobuf_server.py` file to implement the new `assignGrade` method in the `School` class.
4.  **Restart the Kitchen**: Stop your server (Ctrl+C in the terminal) and start it again with the updated code.
5.  **Place a New Order**: In the client code, call the new `assignGrade` method for Martin and print his new grade.

In [1]:
%%writefile ../schema/person.proto
syntax = "proto3";

message Person {
  string user_name = 1;
  optional int64 favorite_number = 2;
  repeated string interests = 3;
  int32 grade = 4;
}

message CourseRequest {
  Person person = 1;
  string course = 2;
}

message GradeRequest {
  Person person = 1;
  int32 grade = 2;
}

service School {
  rpc teachCourse(CourseRequest) returns (Person) {}
  rpc assignGrade(GradeRequest) returns (Person) {}
}

Overwriting ../schema/person.proto


In [2]:
%%writefile ../person_protobuf_server.py
from concurrent import futures
import grpc
import person_pb2
import person_pb2_grpc

class School(person_pb2_grpc.SchoolServicer):

    def teachCourse(self, request, context):
        request.person.interests.append(request.course)
        return request.person

    def assignGrade(self, request, context):
        request.person.grade = request.grade
        return request.person

server = grpc.server(futures.ThreadPoolExecutor(max_workers=2))
person_pb2_grpc.add_SchoolServicer_to_server(School(), server)
server.add_insecure_port('[::]:50051')
print("Starting and running the server...")
print("Press Ctrl+C to stop the server.")
server.start()
server.wait_for_termination()

Overwriting ../person_protobuf_server.py


In [3]:
import sys
sys.path.append('..')
import grpc
import person_pb2
import person_pb2_grpc

with grpc.insecure_channel('localhost:50051') as channel:
    stub = person_pb2_grpc.SchoolStub(channel)
    
    martin = person_pb2.Person(user_name='Martin', favorite_number=1337, interests=["daydreaming", "hacking"])
    
    martin = stub.teachCourse(person_pb2.CourseRequest(person=martin, course="coding"))
    print(martin.interests)
    
    martin = stub.assignGrade(person_pb2.GradeRequest(person=martin, grade=75))
    print(martin.grade)

['daydreaming', 'hacking', 'coding']
75


Once you are done with the server, go to the terminal and press `Ctrl+C` to terminate the server.